In [ ]:
import kagglehub
import os

In [ ]:
path = kagglehub.dataset_download("sarbagyashakya/basketball-51-dataset")
print("Dataset path:", path)

100%|██████████| 6.02G/6.02G [07:03<00:00, 15.3MB/s]

Extracting files...


Dataset path: /root/.cache/kagglehub/datasets/sarbagyashakya/basketball-51-dataset/versions/1


In [ ]:
base_path = os.path.join(path, os.listdir(path)[0])
class_folders = [f for f in os.listdir(base_path) if not f.startswith('.')]

print("Classes:", class_folders)

Classes: ['3p1', 'ft0', 'ft1', 'mp1', '2p0', '3p0', 'mp0', '2p1']


In [ ]:
import tensorflow as tf
from tensorflow.keras import mixed_precision

In [ ]:
mixed_precision.set_global_policy('mixed_float16')

In [ ]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

print("Using:", gpus)

Using: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
import cv2
import numpy as np
from tensorflow.keras import layers, models
from tensorflow.keras.applications.resnet50 import preprocess_input

In [ ]:
IMG_SIZE = 128
MAX_FRAMES = 6

In [ ]:
label_map = {
    "2p1":0, "2p0":1,
    "3p1":2, "3p0":3,
    "mp1":4, "mp0":5,
    "ft1":6, "ft0":7
}

inv_map = {v:k for k,v in label_map.items()}

In [ ]:
def extract_frames(video_path, label, max_frames=MAX_FRAMES):
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        return [], []

    frames = []
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames == 0:
        return [], []

    step = max(1, total_frames // max_frames)

    count = 0
    i = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if i % step == 0:
            frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
            frame = preprocess_input(frame.astype(np.float32))
            frames.append(frame)
            count += 1

        i += 1
        if count >= max_frames:
            break

    cap.release()
    return frames, [label]*len(frames)

In [ ]:
X, y = [], []

In [ ]:
for class_name in class_folders:
    class_path = os.path.join(base_path, class_name)
    videos = os.listdir(class_path)

    print(f"{class_name}: {len(videos)} videos")

    for vid in videos:
        vid_path = os.path.join(class_path, vid)

        frames, labels = extract_frames(vid_path, label_map[class_name])

        if len(frames) == 0:
            continue

        X.extend(frames)
        y.extend(labels)

3p1: 1178 videos
ft0: 566 videos
ft1: 1724 videos
mp1: 558 videos
2p0: 1418 videos
3p0: 2132 videos
mp0: 785 videos
2p1: 1950 videos


In [ ]:
X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int32)

In [ ]:
dataset = tf.data.Dataset.from_tensor_slices((X, y))
dataset = dataset.shuffle(5000).batch(64).prefetch(tf.data.AUTOTUNE)

In [ ]:
base_model = tf.keras.applications.ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = False

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step


In [ ]:
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.5)(x)
output = layers.Dense(8, activation='softmax', dtype='float32')(x)

In [ ]:
model = models.Model(inputs=base_model.input, outputs=output)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
model.fit(dataset, epochs=5)

Epoch 1/5
967/967 ━━━━━━━━━━━━━━━━━━━━ 39s 16ms/step - accuracy: 0.4970 - loss: 1.7470
Epoch 2/5
967/967 ━━━━━━━━━━━━━━━━━━━━ 9s 10ms/step - accuracy: 0.5177 - loss: 1.6171
Epoch 3/5
967/967 ━━━━━━━━━━━━━━━━━━━━ 9s 10ms/step - accuracy: 0.5425 - loss: 1.4666
Epoch 4/5
967/967 ━━━━━━━━━━━━━━━━━━━━ 9s 10ms/step - accuracy: 0.5647 - loss: 1.3674
Epoch 5/5
967/967 ━━━━━━━━━━━━━━━━━━━━ 9s 10ms/step - accuracy: 0.5839 - loss: 1.2981


In [ ]:
for layer in base_model.layers[-100:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(dataset, epochs=5)

Epoch 1/5
967/967 ━━━━━━━━━━━━━━━━━━━━ 110s 54ms/step - accuracy: 0.4121 - loss: 1.7765
Epoch 2/5
967/967 ━━━━━━━━━━━━━━━━━━━━ 21s 22ms/step - accuracy: 0.5078 - loss: 1.4372
Epoch 3/5
967/967 ━━━━━━━━━━━━━━━━━━━━ 20s 21ms/step - accuracy: 0.5949 - loss: 1.1965
Epoch 4/5
967/967 ━━━━━━━━━━━━━━━━━━━━ 20s 21ms/step - accuracy: 0.6478 - loss: 1.0434
Epoch 5/5
967/967 ━━━━━━━━━━━━━━━━━━━━ 20s 21ms/step - accuracy: 0.6925 - loss: 0.9133


In [ ]:
def predict_single_video(video_path, model):
    frames, _ = extract_frames(video_path, label=0)

    if len(frames) == 0:
        print("No frames extracted")
        return None, None

    frames = np.array(frames, dtype=np.float32)

    preds = model.predict(frames)
    avg_pred = np.mean(preds, axis=0)
    avg_pred = avg_pred / np.sum(avg_pred)

    final_class = np.argmax(avg_pred)
    confidence = np.max(avg_pred)

    return final_class, confidence

In [ ]:
def compute_stats(counts):
    FG = (counts["mp1"] / max(1, counts["mp0"] + counts["mp1"])) * 100
    TP = (counts["3p1"] / max(1, counts["3p0"] + counts["3p1"])) * 100
    FT = (counts["ft1"] / max(1, counts["ft0"] + counts["ft1"])) * 100

    FGA = counts["mp0"] + counts["mp1"] + counts["3p0"] + counts["3p1"]

    eFG = (counts["mp1"] + counts["3p1"] + 0.5*counts["3p1"]) / max(1, FGA)

    points = 2*counts["mp1"] + 3*counts["3p1"] + counts["ft1"]
    FTA = counts["ft0"] + counts["ft1"]

    TS = points / max(1, 2*(FGA + 0.44*FTA))

    return {
        "counts": counts,
        "FG%": FG,
        "3P%": TP,
        "FT%": FT,
        "eFG%": eFG,
        "TS%": TS
    }

In [ ]:
def single_shot_stats(pred_label):
    counts = {k:0 for k in label_map.keys()}
    counts[pred_label] = 1
    return compute_stats(counts)

In [ ]:
test_class = class_folders[0]
test_folder = os.path.join(base_path, test_class)

In [ ]:
test_video = os.listdir(test_folder)[0]
test_video_path = os.path.join(test_folder, test_video)

In [ ]:
pred_class, confidence = predict_single_video(test_video_path, model)

1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step


In [ ]:
print(f"Prediction: {inv_map[pred_class]}")
print(f"Confidence: {confidence:.4f}")

Prediction: 2p1
Confidence: 0.8415


In [ ]:
stats = single_shot_stats(inv_map[pred_class])

In [ ]:
print("\n Stats:")
for k, v in stats.items():
    print(k, ":", v)


 Stats:
counts : {'2p1': 1, '2p0': 0, '3p1': 0, '3p0': 0, 'mp1': 0, 'mp0': 0, 'ft1': 0, 'ft0': 0}
FG% : 0.0
3P% : 0.0
FT% : 0.0
eFG% : 0.0
TS% : 0.0


In [ ]:
import kagglehub

# Download latest version for lfw
path = kagglehub.dataset_download("jessicali9530/lfw-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'lfw-dataset' dataset.
Path to dataset files: /kaggle/input/lfw-dataset


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("trainingdatapro/basketball-tracking-dataset")

print("Path to dataset files:", path)

100%|██████████| 200M/200M [00:12<00:00, 16.3MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/trainingdatapro/basketball-tracking-dataset/versions/3


In [ ]:
model.save("basketball_shot_classifier.h5")
print("Model saved as basketball_shot_classifier.h5")

Model saved as basketball_shot_classifier.h5


In [ ]:
from google.colab import files
files.download("basketball_shot_classifier.h5")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>